In [1]:
import pandas as pd

df = pd.read_csv('cleaned_data.csv')

for col in df.columns:
    print(col) 

Date of Surgery
Sex
Age
LLIF?
Case/Type of Surgery
Perc screws?
Open
Open Check V2
Standalone XLIF Check
 Retroperitoneal Approach (LLIF ± ALIF)
Anterior + Posterior Apporoach
Osteotomies (yes/no)
osteotomy level
T12-L1
L1-L2
L2-L3
L3-L4
L4-L5
L5-S1
ALIF Count
Lateral Count
ACR (y=1)
ACR level
Additional Procedures w/in surgery
Pre op Diagnosis (back pain, adjacent segment disease, spondy)
BMI
prior back surgeries? (y=1)
Average PI
VALIDATION COLUMN
PI-LL angle mismatch
ABS PI-LL angle mismatch
PI-LL Mismatch Category (1 = mismatch > +/- 9
PI-LL Mismatch Category (1 = mismatch > +/- 10
(1 = PI>50)
Post-op SS
Post-op SS Grouping
Roussouly Class
post PI
post PT
post LL
post SVA
post-op date
most recent PI
most recent PT
most recent LL
most recent SVA
most recent date
subsidence 
titanium v PEEK (T=0,P=1)
cage details
length of hospital stay (d)
post op complications? (Y/N)
infection 1=yes
DVT  1=yes
PE  1=yes
MI 1=yes
infection 
DVT
PE
MI
Hernia
Hematoma
femoral palsy (knee extension wea

In [2]:
clean_feature_cols = [

    # Demographics
    "Sex",
    "Age",
    "BMI",
    "prior back surgeries? (y=1)",

    # Pre-op diagnosis
    "dx_adjacent_segment",
    "dx_spondylolisthesis",
    "dx_spondylosis",
    "dx_stenosis",
    "dx_scoliosis",
    "dx_flat_back",
    "dx_sagittal_imbalance",
    "dx_post_laminectomy",
    "dx_deformity",

    # Case type
    "Case/Type of Surgery",
    "Additional Procedures w/in surgery",

    # Fusion levels
    "T12-L1",
    "L1-L2",
    "L2-L3",
    "L3-L4",
    "L4-L5",
    "L5-S1",

    # Surgical approach / technique
    "Perc screws?",
    "Open",
    "Open Check V2",
    "Standalone XLIF Check",
    "Retroperitoneal Approach (LLIF ± ALIF)",
    "Anterior + Posterior Apporoach",
    "Osteotomies (yes/no)",
    "osteotomy level",
    "ALIF Count",
    "Lateral Count",
    "ACR (y=1)",
    "ACR level",

    # Preoperative spinopelvic parameters
    "Average PI",
    "PI-LL angle mismatch",
    "ABS PI-LL angle mismatch",
    "PI-LL Mismatch Category (1 = mismatch > +/- 9",
    "PI-LL Mismatch Category (1 = mismatch > +/- 10",
    "(1 = PI>50)",

    # Immediate postoperative alignment (allowed)
    "Post-op SS",
    "post PI",
    "post PT",
    "post LL",
    "post SVA",

    # Immediate postop complications (allowed)
    "infection 1=yes",
    "DVT  1=yes",
    "PE  1=yes",
    "MI 1=yes",
    "femoral palsy (knee extension weakness) 1=yes",
    "hip flexion weakness (iliopsoas weakness)  1=yes",
    "acute thigh paresthesia (immediate post op)",
    "psoas hematoma",

    # Hospital course
    "length of hospital stay (d)",
]


In [3]:
clean_feature_cols_asd = [

    # -----------------
    # Demographics
    # -----------------
    "Sex",
    "Age",
    "BMI",
    "prior back surgeries? (y=1)",

    # -----------------
    # Pre-op diagnosis flags (from Pre op Diagnosis text)
    # -----------------
    "dx_adjacent_segment",
    "dx_spondylolisthesis",
    "dx_spondylosis",
    "dx_stenosis",
    "dx_scoliosis",
    "dx_flat_back",
    "dx_sagittal_imbalance",
    "dx_post_laminectomy",
    "dx_deformity",
    "Case/Type of Surgery",


    # -----------------
    # Fusion levels (index constructs)
    # -----------------
    "T12-L1",
    "L1-L2",
    "L2-L3",
    "L3-L4",
    "L4-L5",
    "L5-S1",

    # Construct summary features you engineered
    "levels_fused_count",
    "construct_span_levels",
    "thoracolumbar_junction",
    "upper_lumbar",
    "lower_lumbar",
    "lumbosacral",

    # -----------------
    # Surgical approach / technique
    # -----------------
    "LLIF?",                # whether lateral/LLIF was done at all
    "Perc screws?",
    "Open",
    "Open Check V2",
    "Standalone XLIF Check",
    "Retroperitoneal Approach (LLIF ± ALIF)",
    "Anterior + Posterior Apporoach",
    "Osteotomies (yes/no)",
    "osteotomy level",
    "ALIF Count",
    "Lateral Count",
    "ACR (y=1)",
    "ACR level",

    # Higher-level surgery flags
    "revision_surgery",
    "deformity_case_text",
    "llif_or_lateral_text",
    "xlif_text",
    "alif_text",

    # -----------------
    # Preoperative spinopelvic parameters
    # -----------------
    "Average PI",
    "PI-LL angle mismatch",
    "ABS PI-LL angle mismatch",
    "PI-LL Mismatch Category (1 = mismatch > +/- 9",
    "PI-LL Mismatch Category (1 = mismatch > +/- 10",
    "(1 = PI>50)",   # this matches your full column list

    # -----------------
    # Immediate postoperative alignment (non-leaky)
    # -----------------
    "Post-op SS",
    "post PI",
    "post PT",
    "post LL",
    "post SVA",

    # -----------------
    # Immediate postop complications (index hospitalization)
    # -----------------
    "infection 1=yes",
    "DVT  1=yes",
    "PE  1=yes",
    "MI 1=yes",
    "femoral palsy (knee extension weakness) 1=yes",
    "hip flexion weakness (iliopsoas weakness)  1=yes",
    "acute thigh paresthesia (immediate post op)",
    "psoas hematoma",

    # -----------------
    # Hospital course
    # -----------------
    "length of hospital stay (d)",
]


In [4]:
import pandas as pd
import numpy as np
from sksurv.util import Surv
from sklearn.model_selection import train_test_split

time_col = "Time Until ASD Diagnosis (months)"


In [5]:
import seaborn as sns
from pandas.api import types as pd_types

import matplotlib.pyplot as plt

# helper to find column with case-insensitive / partial matching
def find_col(df, name):
    name_l = name.lower()
    # exact lower match
    for c in df.columns:
        if c.lower() == name_l:
            return c
    # substring match
    for c in df.columns:
        if name_l in c.lower() or c.lower() in name_l:
            return c
    return None

# desired columns (use existing notebook variables where available)
cols_wanted = [
    "REVERIFIED ASD",                    # user: 'Reverified ASD'
    time_col,                            # 'Time Until ASD Diagnosis (months)' (variable from notebook)
    "need revision surgery? (Y=1)",      # user-provided name
    "radiographic adjacent segment disease? (Y/N)"
]

# resolve to actual df columns, skip missing
cols = []
for nm in cols_wanted:
    c = find_col(df, nm)
    if c is None:
        print(f"Column not found (skipping): {nm}")
    else:
        cols.append(c)

if not cols:
    raise RuntimeError("No plotting columns found in df.")


In [6]:
import pandas as pd


# 2. Clean column names a bit
df.columns = (
    df.columns
      .str.strip()
      .str.replace("\n", " ", regex=False)
)

df.columns



Index(['Date of Surgery', 'Sex', 'Age', 'LLIF?', 'Case/Type of Surgery',
       'Perc screws?', 'Open', 'Open Check V2', 'Standalone XLIF Check',
       'Retroperitoneal Approach (LLIF ± ALIF)',
       ...
       'alif_text', 'dx_adjacent_segment', 'dx_spondylolisthesis',
       'dx_spondylosis', 'dx_stenosis', 'dx_scoliosis', 'dx_flat_back',
       'dx_sagittal_imbalance', 'dx_post_laminectomy', 'dx_deformity'],
      dtype='str', length=116)

In [7]:
target = "Time Until ASD Diagnosis (months)"

# Keep only rows where the target is known (patients who actually developed ASD)
asd_df = df[df[target].notna()].copy()
asd_df.shape


(125, 116)

In [8]:
# -----------------------------
# Final pipeline: feature prep -> PCA -> RSF (fixed best params) -> metrics + save + unit test
# -----------------------------
import os
import pickle
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.model_selection import StratifiedKFold

from sksurv.util import Surv
from sksurv.ensemble import RandomSurvivalForest
from sksurv.metrics import concordance_index_censored, integrated_brier_score, cumulative_dynamic_auc
from lifelines import KaplanMeierFitter

# -----------------------------
# 0. CONFIG: best RSF params 
# -----------------------------
BEST_RSFPARAMS = {
    "n_estimators": 200,
    "min_samples_split": 4,
    "min_samples_leaf": 12,
    "max_features": "sqrt",
    "n_jobs": -1,
    "random_state": 42,
}

# -----------------------------
# 1. Survival labels & basic checks (assumes df available)
# -----------------------------
event_col = "REVERIFIED ASD"
time_asd_col = "Time Until ASD Diagnosis (months)"
time_no_asd_col = "Time Without_ASD (months)"

df_surv = df.copy()
df_surv[event_col] = df_surv.get(event_col, 0).fillna(0).astype(int)
df_surv["time_surv"] = np.where(
    df_surv[event_col] == 1,
    df_surv.get(time_asd_col),
    df_surv.get(time_no_asd_col),
)
df_surv = df_surv.dropna(subset=["time_surv"])
df_surv["time_surv"] = df_surv["time_surv"].astype(float)

print("Cohort rows:", df_surv.shape[0], "Events:", int(df_surv[event_col].sum()))

# -----------------------------
# 2. Feature construction 
# -----------------------------
# clean_feature_cols must exist (list of features you'd like to start from)
X = df_surv[clean_feature_cols].copy()

numeric_like_cols = [
    "BMI",
    "ALIF Count",
    "Lateral Count",
    "Average PI",
    "PI-LL angle mismatch",
    "ABS PI-LL angle mismatch",
    "Post-op SS",
    "post PI",
    "post PT",
    "post LL",
    "post SVA",
    "length of hospital stay (d)",
]

# coerce numeric-like
for col in numeric_like_cols:
    if col in X.columns:
        X[col] = pd.to_numeric(X[col], errors="coerce")

# one-hot encode categoricals
cat_cols = X.select_dtypes(include=["object", "category"]).columns.tolist()
print("Categorical feature columns:", cat_cols)
X_encoded = pd.get_dummies(X, columns=cat_cols, drop_first=True)

# simple imputation
X_encoded = X_encoded.fillna(0.0)

# targets
durations_all = df_surv["time_surv"].astype(float).values
events_all = df_surv[event_col].astype(bool).values

print("Initial feature matrix shape:", X_encoded.shape)

# -----------------------------
# 3. Remove ultra-rare binary features (occurence < 5)
# -----------------------------
binary_like_cols = [c for c in X_encoded.columns if set(np.unique(X_encoded[c])) <= {0,1}]
rare_cols = [c for c in binary_like_cols if X_encoded[c].sum() < 5]
print(f"Dropping {len(rare_cols)} ultra-rare binary cols.")
X_reduced = X_encoded.drop(columns=rare_cols)
print("Shape after dropping ultra-rare columns:", X_reduced.shape)

# -----------------------------
# 4. PCA compression 
# -----------------------------
numeric_present = [c for c in numeric_like_cols if c in X_reduced.columns]
print("Numeric-like columns for PCA:", numeric_present)

if len(numeric_present) >= 2:
    scaler_for_pca = StandardScaler()
    numeric_data = scaler_for_pca.fit_transform(X_reduced[numeric_present].values)
    n_pca = min(3, numeric_data.shape[1])
    pca = PCA(n_components=n_pca, random_state=42)
    pcs = pca.fit_transform(numeric_data)
    pca_cols = [f"pca_num_{i+1}" for i in range(n_pca)]
    df_pcs = pd.DataFrame(pcs, columns=pca_cols, index=X_reduced.index)
    X_reduced = pd.concat([X_reduced.drop(columns=numeric_present), df_pcs], axis=1)
    print(f"Replaced {len(numeric_present)} numeric cols with {n_pca} PCA components.")
else:
    print("Skipping PCA (not enough numeric-like columns).")

print("Final feature matrix shape (X_reduced):", X_reduced.shape)

import pickle
import numpy as np
import pandas as pd
from sksurv.metrics import concordance_index_censored
from sksurv.util import Surv

# Load trained model
with open("models/rsf_final_best.pkl", "rb") as f:
    bundle = pickle.load(f)

rsf = bundle["model"]
feature_cols = bundle["feature_columns"]


Cohort rows: 546 Events: 122
Categorical feature columns: ['Case/Type of Surgery', 'Additional Procedures w/in surgery', 'osteotomy level', 'ACR level']
Initial feature matrix shape: (546, 424)
Dropping 354 ultra-rare binary cols.
Shape after dropping ultra-rare columns: (546, 70)
Numeric-like columns for PCA: ['BMI', 'ALIF Count', 'Lateral Count', 'Average PI', 'PI-LL angle mismatch', 'ABS PI-LL angle mismatch', 'Post-op SS', 'post PI', 'post PT', 'post LL', 'post SVA', 'length of hospital stay (d)']
Replaced 12 numeric cols with 3 PCA components.
Final feature matrix shape (X_reduced): (546, 61)


C:\Users\shrin\AppData\Local\Temp\ipykernel_7488\1042527084.py:78: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_cols = X.select_dtypes(include=["object", "category"]).columns.tolist()


In [9]:
# Robust unit test for RSF survival predictions (replacement for failing assertion)
import pickle
import numpy as np
import pandas as pd
import os

model_path = "models/rsf_final_best.pkl"
assert os.path.exists(model_path), f"Model file not found: {model_path}"

with open(model_path, "rb") as f:
    bundle = pickle.load(f)

model = bundle["model"]
cols = bundle["feature_columns"]

# Build synthetic sample (median of training features if available)
FEATURE_MATRIX = globals().get('X_reduced', globals().get('X_encoded', None))
if FEATURE_MATRIX is not None:
    try:
        x0 = FEATURE_MATRIX.median(axis=0).to_numpy().reshape(1, -1)
        # If one-hot encoded columns are fractional due to median, round the categorical ones:
        # set any column whose unique values are only 0/1 in training to nearest {0,1}
        for i, colname in enumerate(FEATURE_MATRIX.columns):
            unique_vals = FEATURE_MATRIX[colname].dropna().unique()
            if set(np.unique(unique_vals)).issubset({0,1}):
                x0[0, i] = 1.0 if x0[0, i] >= 0.5 else 0.0
    except Exception:
        x0 = np.zeros((1, len(cols)))
else:
    # fallback: zeros, but must match number of model columns
    x0 = np.zeros((1, len(cols)))

# Try both return array and return functions
surv_out = None
# Preferred: ask for array
try:
    surv_arr = model.predict_survival_function(x0, return_array=True)
    # model may return a 2D array or list; normalize to numpy array
    if isinstance(surv_arr, (list, tuple)):
        surv_arr = np.asarray(surv_arr)
    surv_out = np.array(surv_arr)  # shape (n_samples, n_times)
    times_grid = None
    # If shape is (n_samples, n_times), we'll create a default candidate time grid 0..max training time (approx)
except TypeError:
    # some implementations don't accept return_array arg or expect different signature
    try:
        funcs = model.predict_survival_function(x0, return_array=False)
        # funcs might be a single function or list of functions
        if not isinstance(funcs, (list, tuple)):
            funcs = [funcs]
        # build a time-grid by unioning xs from functions
        xs_union = np.unique(np.concatenate([np.asarray(f.x) for f in funcs]))
        times_grid = np.sort(xs_union)
        surv_out = np.zeros((len(funcs), len(times_grid)))
        for i, f in enumerate(funcs):
            surv_out[i, :] = np.interp(times_grid, np.asarray(f.x), np.asarray(f.y), left=1.0, right=f.y[-1])
    except Exception as ex:
        raise AssertionError(f"predict_survival_function failed entirely: {ex}")

# Now surv_out should be a numpy array shape (n_samples, n_times)
if surv_out is None:
    raise AssertionError("Could not obtain survival array from model.")

# Basic shape checks
n_samples, n_times = surv_out.shape
assert n_samples == 1, f"Expected single sample, got {n_samples}"

# Evaluate monotonicity (non-increasing) and bounds for the single sample
vals = surv_out[0, :]

# Monotonic non-increasing check (allow tiny numerical tolerance)
if not np.all(np.diff(vals) <= 1e-6):
    # print first few violations for debugging
    diffs = np.diff(vals)
    viol = np.where(diffs > 1e-6)[0]
    raise AssertionError(f"Survival function not non-increasing: first violation index {viol[0]} (diff={diffs[viol[0]]})")

# Bound check
if np.any(vals < -1e-6) or np.any(vals > 1.0 + 1e-6):
    raise AssertionError("Survival probabilities out of [0,1] bounds")

# Risk score check (must be finite)
risk = model.predict(x0)
if not np.all(np.isfinite(risk)):
    raise AssertionError("Risk score contains non-finite values")

print("Unit test PASSED.")
print("Risk score:", risk)
# Print a small summary of the survival function
if 'times_grid' in locals() and times_grid is not None:
    print("Survival evaluated at times (first 6):", list(zip(times_grid[:6], vals[:6])))
else:
    print("Survival values (first 6):", vals[:6])


Unit test PASSED.
Risk score: [10.74922231]
Survival values (first 6): [0.99525318 0.97149044 0.95456243 0.91531089 0.90830732 0.89868999]


In [10]:
# -------- Test 1: Low-risk vs High-risk patient --------
import numpy as np

x_base = x0.copy()

# Predict survival
S_base = model.predict_survival_function(x_base, return_array=True)[0]

# -------- Test 2: S(t) interpretation check --------

# Choose a horizon (e.g., 12 months)
t_idx = min(12, len(S_base)-1)

S12 = S_base[t_idx]
P_event_12 = 1 - S12

print(f"S(12 mo) = {S12:.3f}")
print(f"P(ASD by 12 mo) = {P_event_12:.3f}")

assert 0 <= S12 <= 1
assert 0 <= P_event_12 <= 1
assert abs(S12 + P_event_12 - 1) < 1e-6

print("TEST 2 PASSED: survival and event probability consistent.")


S(12 mo) = 0.801
P(ASD by 12 mo) = 0.199
TEST 2 PASSED: survival and event probability consistent.


In [11]:
# -------- Test: Risk ordering, not survival mean --------

x_base = x0.copy()
x_high = x0.copy()

def bump_pca(name, delta):
    if name in FEATURE_MATRIX.columns:
        idx = FEATURE_MATRIX.columns.get_loc(name)
        x_high[0, idx] += delta

# Push PCA in any direction (sign does NOT matter)
bump_pca("pca_num_1", +1.0)
bump_pca("pca_num_2", +1.0)
bump_pca("pca_num_3", +1.0)

risk_base = model.predict(x_base)[0]
risk_high = model.predict(x_high)[0]

print("Risk (base):", risk_base)
print("Risk (high):", risk_high)

assert risk_high != risk_base, "Perturbation had no effect on risk"

print("TEST PASSED: PCA perturbation affects risk score.")


Risk (base): 10.749222312595911
Risk (high): 9.468230361206993
TEST PASSED: PCA perturbation affects risk score.


In [12]:
# Optional: population-level risk ordering sanity check

X_pop = X_reduced.sample(100, random_state=42).values
risks = model.predict(X_pop)

# Check that risk has variance (not constant)
assert np.std(risks) > 0, "Risk scores have zero variance"

print("Optional test passed: population risk distribution non-degenerate.")


Optional test passed: population risk distribution non-degenerate.
